# 🛡️ Sudarshana (StegShield) — Account 2 Training Pipeline
### Parallel Ensemble Training (Seed 101) on NVIDIA T4 GPU

This runs on your **Second Google Account** simultaneously with Account 1.
Run cells top-to-bottom: **1 → 2 → 3 → 4 → 5 → 6**.

In [ ]:
# ── Cell 1: Verify T4 GPU Acceleration ──
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected! Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# ── Cell 2: Upload & Extract Codebase ──
import os, sys

if not os.path.exists('/content/EDI_Project_Colab_Latest.zip'):
    from google.colab import files
    print("Please select and upload EDI_Project_Colab_Latest.zip:")
    uploaded = files.upload()
    ZIP_PATH = list(uploaded.keys())[0]
else:
    ZIP_PATH = '/content/EDI_Project_Colab_Latest.zip'

!unzip -q -o "{ZIP_PATH}" -d /content/EDI_Project
%cd /content/EDI_Project
sys.path.insert(0, 'src')
!pip install -q scipy scikit-learn seaborn plotly
print("✅ Workspace ready at /content/EDI_Project")

In [ ]:
# ── Cell 3: Setup Matched Dataset & Cyber Threats ──
import os
import numpy as np
from PIL import Image
from pathlib import Path

os.makedirs("data/clean", exist_ok=True)
os.makedirs("data/stego", exist_ok=True)

# Check if Google Drive dataset is accessible:
DRIVE_CLEAN = "/content/drive/MyDrive/EDI_Project_Final/data/clean"
DRIVE_STEGO = "/content/drive/MyDrive/EDI_Project_Final/data/stego"

if os.path.exists(DRIVE_CLEAN) and os.path.exists(DRIVE_STEGO) and len(os.listdir(DRIVE_CLEAN)) > 0:
    print("⚡ Copying 10,000 images from Google Drive...")
    !cp -r "{DRIVE_CLEAN}"/* data/clean/
    !cp -r "{DRIVE_STEGO}"/* data/stego/
else:
    print("⚡ Generating 300 matched clean and stego training pairs...")
    for i in range(1, 301):
        stem = f"carrier_pair_{i:04d}"
        x = np.linspace(0, 10 + (i % 5), 256)
        y = np.linspace(0, 10 + (i % 7), 256)
        xx, yy = np.meshgrid(x, y)
        texture = np.sin(xx) * np.cos(yy) * 45 + np.random.normal(128, 12, (256, 256))
        clean_arr = np.clip(texture, 0, 255).astype(np.uint8)
        Image.fromarray(clean_arr).save(f"data/clean/{stem}.png")
        
        stego_arr = clean_arr.copy()
        rng = np.random.default_rng(101 + i)
        mask = rng.integers(0, 2, size=(256, 256), dtype=np.uint8)
        stego_arr = (stego_arr & 0xFE) | mask
        Image.fromarray(stego_arr).save(f"data/stego/{stem}.png")

clean_num = len(os.listdir("data/clean"))
stego_num = len(os.listdir("data/stego"))
print(f"\n📊 Dataset Ready: {clean_num} Clean | {stego_num} Stego")

In [ ]:
# ── Cell 4: Start GPU Training (Account 2: Seed 101) ──
!python src/training/train.py \
    --mode joint \
    --epochs 30 \
    --lr 1e-4 \
    --batch_size 32 \
    --seed 101

In [ ]:
# ── Cell 5: Evaluate Threats & Test Accuracy (Run after Cell 4 completes) ──
import torch
from src.model.fusion_model import MultiBranchSteganalyzer
from src.data_pipeline.dataset import get_dataloaders
from src.utils.checkpoints import extract_state_dict

# 1. Evaluate real-world malware interception
!python scripts/generate_malicious_stego.py --evaluate

# 2. Calculate held-out test accuracy
print("\n" + "=" * 60)
print("  Computing Held-out Test Set Accuracy")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_, _, test_loader = get_dataloaders(
    clean_dir="data/clean",
    stego_dir="data/stego",
    batch_size=32
)

model = MultiBranchSteganalyzer().to(device)
raw = torch.load("weights/fusion_best.pt", map_location=device)
model.load_state_dict(extract_state_dict(raw), strict=False)
model.eval()

total_clean_correct, total_clean_samples = 0, 0
total_stego_correct, total_stego_samples = 0, 0

with torch.no_grad():
    for clean_imgs, stego_imgs in test_loader:
        B = clean_imgs.size(0)
        images = torch.cat([clean_imgs, stego_imgs], dim=0).to(device)
        labels = torch.cat([
            torch.zeros(B, dtype=torch.long),
            torch.ones(B, dtype=torch.long)
        ], dim=0).to(device)

        logits = model(images)
        preds = logits.argmax(dim=1)
        clean_preds = preds[:B]
        stego_preds = preds[B:]

        total_clean_correct += (clean_preds == 0).sum().item()
        total_clean_samples += B
        total_stego_correct += (stego_preds == 1).sum().item()
        total_stego_samples += B

total_correct = total_clean_correct + total_stego_correct
total_samples = total_clean_samples + total_stego_samples

clean_acc = (total_clean_correct / total_clean_samples * 100) if total_clean_samples > 0 else 0
stego_acc = (total_stego_correct / total_stego_samples * 100) if total_stego_samples > 0 else 0
overall_acc = (total_correct / total_samples * 100) if total_samples > 0 else 0

print("-" * 60)
print(f"  Clean Cover Accuracy  (TN rate): {clean_acc:6.2f}% ({total_clean_correct}/{total_clean_samples})")
print(f"  Stego Carrier Accuracy (TP rate): {stego_acc:6.2f}% ({total_stego_correct}/{total_stego_samples})")
print(f"  Overall Test Accuracy           : {overall_acc:6.2f}% ({total_correct}/{total_samples})")
print("-" * 60)
print(f"🏆 Account 2 Verdict: Evaluated {total_samples} held-out images.")

In [ ]:
# ── Cell 6: Download & Save Trained Weights (Seed 101) ──
import os, shutil
from google.colab import files

# Copy to custom named checkpoint for ensembling
shutil.copy2('weights/fusion_best.pt', 'weights/fusion_seed101.pt')

# 1. Save to Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy2('weights/fusion_seed101.pt', '/content/drive/MyDrive/fusion_seed101.pt')
    print("✅ Saved backup to Google Drive: fusion_seed101.pt")

# 2. Direct browser download to your Mac:
print("📥 Downloading Account 2 weights (fusion_seed101.pt)...")
files.download('weights/fusion_seed101.pt')